# Wafer Defect Pattern Classification

## Overview

This notebook analyzes the **WM-811K wafer map dataset**, which contains wafer test results from semiconductor manufacturing. The goal is to explore the dataset and apply machine learning techniques to classify wafer defect patterns.

In semiconductor fabrication, each wafer contains many integrated circuits (dies). After manufacturing, the wafer undergoes **electrical probe testing**, where each die is evaluated for functionality. The results of this testing are represented as a **wafer map**, which is a spatial grid indicating whether each chip passed or failed testing.

Certain manufacturing issues produce characteristic spatial patterns of failures. Engineers analyze these patterns to identify potential process issues.

In this project we treat wafer maps as **image-like spatial data** and attempt to classify the type of defect pattern using machine learning techniques.

## Goals

The objectives of this analysis are:

1. Explore and understand the wafer dataset
2. Visualize defect patterns
3. Prepare wafer maps for machine learning
4. Train a baseline model
5. Train a convolutional neural network (CNN)
6. Evaluate model performance

In [2]:
# Core numerical and data libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Deep learning
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

import os
import warnings
warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'sklearn'

## Loading the Dataset

The wafer data is stored as a pickle file containing wafer test results.

Each row represents one wafer and includes:

- waferMap: 2D matrix representing pass/fail dies
- dieSize: die dimensions
- lotName: manufacturing batch identifier
- waferIndex: wafer number
- trianTestLabel: training/test designation
- failureType: defect classification label

In [ ]:
print(os.listdir("X:/ECEsite/Machine-Learning-Class-Repo/"))

In [ ]:
print(os.listdir("X:/ECEsite/Machine-Learning-Class-Repo/"))

In [ ]:
df.info()

The distribution of defect classes is important because class imbalance can affect model performance.

Some defect types occur much more frequently than others.

In [ ]:
plt.figure(figsize=(10,5))

df['failureType'].value_counts().plot(kind='bar')

plt.title("Distribution of Wafer Failure Types")
plt.xlabel("Failure Type")
plt.ylabel("Count")

plt.show()

Each wafer map is stored as a matrix representing the spatial layout of dies on a wafer.

We visualize several examples to understand the types of spatial patterns that exist in the dataset.

In [ ]:
sample = df.iloc[0]['waferMap']

plt.imshow(sample)
plt.title("Example Wafer Map")
plt.colorbar()
plt.show()

In [ ]:
fig, axes = plt.subplots(2,5, figsize=(12,6))

for i, ax in enumerate(axes.flatten()):
    wafer = df.iloc[i]['waferMap']
    label = df.iloc[i]['failureType']

    ax.imshow(wafer)
    ax.set_title(label)
    ax.axis('off')

plt.tight_layout()
plt.show()

Some wafer maps do not have defect labels. Since this project focuses on supervised learning, we remove unlabeled samples.

In [ ]:
df_clean = df[df['failureType'] != 'none']

In [ ]:
df_clean['failureType'].value_counts()

Machine learning models require numerical arrays.

We convert the wafer maps into numpy arrays and extract the corresponding defect labels.

In [ ]:
print(X.shape)

In [ ]:
le = LabelEncoder()

y_encoded = le.fit_transform(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42
)

As a baseline, we use a classical machine learning model.

Because traditional models require flat feature vectors, we reshape each wafer map into a one-dimensional vector.

In [ ]:
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

In [ ]:
rf = RandomForestClassifier(n_estimators=100)

rf.fit(X_train_flat, y_train)

preds = rf.predict(X_test_flat)

In [ ]:
print(classification_report(y_test, preds))

Wafer maps are spatial data similar to images.

Convolutional Neural Networks (CNNs) are well suited for learning spatial patterns, such as rings, scratches, and clusters that appear in wafer defect maps.

In [ ]:
X_train = X_train.reshape(-1,26,26,1)
X_test = X_test.reshape(-1,26,26,1)

In [ ]:
model = Sequential()

model.add(Conv2D(32,(3,3),activation='relu',input_shape=(26,26,1)))
model.add(MaxPooling2D((2,2)))

model.add(Conv2D(64,(3,3),activation='relu'))
model.add(MaxPooling2D((2,2)))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dense(len(le.classes_),activation='softmax'))

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

In [ ]:
model.evaluate(X_test, y_test)

In [ ]:
preds = model.predict(X_test)

preds = np.argmax(preds, axis=1)

In [ ]:
cm = confusion_matrix(y_test, preds)

plt.figure(figsize=(8,6))

sns.heatmap(cm, annot=True, cmap='Blues')

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

## Discussion

The machine learning models demonstrate the ability to classify wafer defect patterns based on spatial structure.

The baseline model provides a simple reference point, while the convolutional neural network improves performance by learning spatial features directly from the wafer maps.

Future improvements could include:

- Data augmentation
- Deeper CNN architectures
- Class imbalance handling
- Feature engineering based on wafer geometry

Understanding defect patterns can help engineers identify manufacturing process issues and improve semiconductor yield.